In [2]:
import os
from typing import Dict, Any, List
import docx

from docx.table import Table
from docx.text.paragraph import Paragraph



In [39]:
import os
import re
import docx
from docx.table import Table
from docx.text.paragraph import Paragraph
from docx.document import Document as DocType
from docx.oxml.table import CT_Tbl
from docx.oxml.text.paragraph import CT_P

# 1. 블록 아이템을 순서대로 읽어오는 함수 (핵심!)
def iter_block_items(parent):
    parent_elm = parent.element.body if isinstance(parent, DocType) else parent.element
    for child in parent_elm.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, parent)
        elif isinstance(child, CT_Tbl):
            yield Table(child, parent)

# 2. 파싱 함수
def parse_docx_clean(file_path: str, lines_per_page=30):
    doc = docx.Document(file_path)
    all_lines = []
    
    # 여기서 직접 함수 호출!
    for block in iter_block_items(doc):
        if isinstance(block, Paragraph):
            text = clean_text(block.text)
            if text: all_lines.append(text)
        elif isinstance(block, Table):
            for row in block.rows:
                row_text = "| " + " | ".join([clean_text(cell.text) for cell in row.cells if clean_text(cell.text)]) + " |"
                all_lines.append(row_text)
                
    pages = []
    for i in range(0, len(all_lines), lines_per_page):
        pages.append({
            "page_number": (i // lines_per_page) + 1,
            "text": "\n".join(all_lines[i:i + lines_per_page])
        })
    return pages

# 3. 테스트 실행
file_path = r"C:\skn24\수업자료\08_large_language_model\00.final_project\튜토리얼\data\RFP\TS 자동차분야 데이터 활용 플랫폼 구축 용역 RFP.docx"
pages = parse_docx_clean(file_path)

for p in pages[:20]:
    print(f"\n--- 페이지 {p['page_number']} ---")
    print(p['text'][:300])


--- 페이지 1 ---
|  |
|  |
| 「TS 자동차분야 데이터 활용 플랫폼 구축」 과업 및 제안요청서 |
|  |
2026. 05.
|  |
| 기획본부 AI디지털실 AI혁신처 |
| Ⅰ | 사업 개요 |
1. 제안 개요
ㅇ (사 업 명) TS 자동차분야 데이터 활용 플랫폼 구축
ㅇ (사업예산) 287,190,000원(부가세 포함)
ㅇ (사업기간) 착수일로부터 2026.12.18.까지
ㅇ (계약방식) 제한경쟁입찰, 협상에 의한 계약(기술 90%, 가격 10%)
2. 추진 배경 및 필요성
① (대외적) 국정과제「국정운영 5개년 계획」의 20번과제 실

--- 페이지 2 ---
* 공단에서 운영 중인 자동차 분야 관련 시스템 일체
ㅇ 비정형데이터의 라벨링 처리 및 데이터의 논리적 분리* 체계 수립
※ 비정형데이터(파일‧영상‧이미지 등)의 저장 영역을 원본, 저장 경로 등으로 분리 관리 환경 설계
ㅇ 공단 전반에 산재되어있는 비정형데이터 메타데이터 확보
ㅇ 주기적으로 활용되는 통계성 데이터셋 목록 사전 구축
ㅇ 데이터 연계부터 활용까지 전주기 이력 관리 프로세스 수립
ㅇ 공단 필요 소프트웨어 설치를 위한 기술 지원
ㅇ AI 서비스 발굴 확대를 위한 외부 연계 파이프라인 환경 사전 구현
□ 공단 데이터 활용 

--- 페이지 3 ---
2. 추진 체계
ㅇ 추진 조직도
|  |
ㅇ 조직 구성별 역할
| 구 분 | 주요 역할 |
| AI혁신처 (사업주관) | ㅇ 사업 기본계획 수립 및 발주 ㅇ 사업관리 및 업무 수행 ㅇ 과업지시서 사전 검토 및 과업심의위원회 추진 ㅇ 프로젝트 추진 방향 검증 및 사업 수행 중대 사안에 대한 의사결정 ㅇ 용역사 정보보안 업무 수행 등 ㅇ 감독 및 검사 |
| 자동차분야 데이터 소관부서 (사업지원) | ㅇ 활용 및 AI학습 정형‧비정형 데이터 선별 ㅇ 통계성 데이터 셋 확보를 위한 데이터 추출 지원 ㅇ (원천시스템 보유 부서) 시스템 간

--- 페이지 4 ---
| - 상용 소프트웨어 설치 |
| - 비정형데이터 체계 수립 |


In [40]:
from pydantic import BaseModel, Field
from typing import List, Optional

class CanonicalRequirement(BaseModel):
    requirement_id: str = Field(..., description="요구사항 고유 ID")
    requirement_name: str = Field(..., description="요구사항명")
    requirement_type: str = Field(..., description="'기능' 또는 '비기능'")
    description: str = Field(..., description="요구사항의 상세 설명 본문")
    source: List[str] = Field(..., description="출처 문서")
    constraints: List[str] = Field(default=[], description="제약사항 목록")
    priority: str = Field(..., description="중요도")
    validation_criteria: List[str] = Field(..., description="검수 및 승인 기준 목록")
    note: Optional[str] = Field(None, description="비고 사항")

class RequirementDocument(BaseModel):
    requirements: List[CanonicalRequirement]

In [41]:
def MAP(page_data: dict, source_name: str) -> List[CanonicalRequirement]:
    text = page_data["text"]
    lines = text.split('\n')
    requirements = []
    current_req = None
    
    for line in lines:
        if not line.strip().startswith('|'): continue
        # 표의 셀들을 리스트로 분리
        parts = [p.strip() for p in line.strip('|').split('|')]
        
        # 1. ID 발견 시 새 객체 생성
        id_match = next((p for p in parts if re.search(r"[A-Za-z]{3}-\d+", p)), None)
        if id_match:
            current_req = CanonicalRequirement(
                requirement_id=id_match,
                requirement_name="확인 중...",
                requirement_type="기능",
                description="",
                source=[f"{source_name}_p{page_data['page_number']}"],
                constraints=[],
                priority="중",
                validation_criteria=[],
                note=None
            )
            requirements.append(current_req)
            
        elif current_req:
            # 2. 행 전체를 문자열로 합쳐서 키워드 찾기
            line_str = " ".join(parts)
            
            # 명칭 추출: '정의' 키워드가 포함된 셀 찾기
            for i, part in enumerate(parts):
                if "정의" in part and i + 1 < len(parts):
                    # '정의'라는 글자를 걷어내고 옆 셀 값을 명칭으로 지정
                    name_val = part.replace("정의", "").strip()
                    if not name_val: name_val = parts[i+1] # 글자가 없으면 옆 셀 사용
                    
                    if current_req.requirement_name == "확인 중...":
                        current_req.requirement_name = name_val.strip(": ")
            
            # 3. 상세내용/세부내용 추출
            if any(k in line_str for k in ["상세설명", "세부내용"]):
                # 키워드가 포함된 셀을 제외한 나머지 셀들을 description에 병합
                content = " ".join([p for p in parts if p and not any(k in p for k in ["상세설명", "세부내용", "정의", "요구사항"])])
                if content:
                    current_req.description += content.strip() + " "

    return requirements

In [42]:
all_reqs = run_extraction_pipeline(file_path)

추출 완료! 저장 경로: C:\skn24\수업자료\08_large_language_model\00.final_project\튜토리얼\data\RFP\TS 자동차분야 데이터 활용 플랫폼 구축 용역 RFP_requirements.json
총 174개의 요구사항이 추출되었습니다.


In [19]:
pages = parse_docx_clean(file_path)
print(f"총 페이지 수: {len(pages)}")
print(f"첫 번째 페이지의 텍스트 앞부분: {pages[0]['text'][:200]}")

# 패턴 강제 테스트
test_text = pages[0]['text']
matches = re.findall(r"\[[A-Za-z]{3}-\d+\]", test_text)
print(f"패턴 매칭된 ID 리스트: {matches}")

총 페이지 수: 33
첫 번째 페이지의 텍스트 앞부분: |  |
|  |
| 「TS 자동차분야 데이터 활용 플랫폼 구축」 과업 및 제안요청서 |
|  |
2026. 05.
|  |
| 기획본부 AI디지털실 AI혁신처 |
| Ⅰ | 사업 개요 |
1. 제안 개요
ㅇ (사 업 명) TS 자동차분야 데이터 활용 플랫폼 구축
ㅇ (사업예산) 287,190,000원(부가세 포함)
ㅇ (사업기간) 착수일로부터 2026.1
패턴 매칭된 ID 리스트: []
